In [34]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from typing_extensions import NotRequired
from langchain_ollama import ChatOllama


In [2]:
model = ChatOllama(model="llama3.2:latest")

In [35]:
#define state
class BatsmanState(TypedDict):
    
    runs: int
    balls: int
    fours: int
    sixes: int
    
    sr: NotRequired[float]
    bpb: NotRequired[float]
    boundry_percent: NotRequired[float]
    summary: NotRequired[str]

In [36]:
def cal_sr(state:BatsmanState):
    
    sr = (state['runs']/state['balls'])*100
    
    return {'sr' : sr}

In [37]:
def cal_bpb(state:BatsmanState):
    
    bpb = state['balls']/(state['fours'] + state['sixes'])
    
    return {'bpb' : bpb}

In [38]:
def cal_boundry_percent(state:BatsmanState):
    
    boundry_percent = ((state['fours']*4 + state['sixes']*6)/state['runs'])*100
    
    return {'boundry_percent' : boundry_percent}

In [39]:
def summary(state: BatsmanState):
    
    summary = f"""
Strike Rate - {state['sr']} \n
Balls per Boundry - {state['bpb']} \n
Boundry Percent - {state['boundry_percent']} \n
"""

    return {'summary' : summary}

In [40]:
#define graph

graph = StateGraph(BatsmanState)

#add nodes

graph.add_node('cal_sr', cal_sr)
graph.add_node('cal_bpb', cal_bpb)
graph.add_node('cal_boundry_percent', cal_boundry_percent)
graph.add_node('summary', summary)

#add edges

graph.add_edge(START,'cal_sr')
graph.add_edge(START,'cal_bpb')
graph.add_edge(START,'cal_boundry_percent')


graph.add_edge('cal_sr', 'summary')
graph.add_edge('cal_bpb', 'summary')
graph.add_edge('cal_boundry_percent', 'summary')

graph.add_edge('summary', END)



In [41]:
workflow = graph.compile()

In [42]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 10,
    'sixes' : 5
}

workflow.invoke(initial_state)


{'runs': 100,
 'balls': 50,
 'fours': 10,
 'sixes': 5,
 'sr': 200.0,
 'bpb': 3.3333333333333335,
 'boundry_percent': 70.0,
 'summary': '\nStrike Rate - 200.0 \n\nBalls per Boundry - 3.3333333333333335 \n\nBoundry Percent - 70.0 \n\n'}